In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.modules.pop("data", None)

import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim

from data import CIFData, collate_pool, get_train_val_test_loader
from model import CrystalGraphConvNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


In [ ]:
# Confirm that Python is importing the local tutorial files.
import data, model as cgcnn_model
print(data.__file__)
print(cgcnn_model.__file__)


In [ ]:
# Start with the small local classification dataset before trying a full regression dataset.
dataset = CIFData(str(PROJECT_ROOT / "Data" / "sample-classification"))
print("Number of crystals:", len(dataset))


In [ ]:
collate_fn = collate_pool
train_loader, val_loader, test_loader = get_train_val_test_loader(
    dataset=dataset,
    collate_fn=collate_fn,
    batch_size=8,
    train_ratio=0.6,
    val_ratio=0.2,
    test_ratio=0.2,
    train_size=None,
    val_size=None,
    test_size=None,
    return_test=True,
    num_workers=0,
)


In [ ]:
structures, _, _ = dataset[0]
orig_atom_fea_len = structures[0].shape[-1]
nbr_fea_len = structures[1].shape[-1]

model = CrystalGraphConvNet(
    orig_atom_fea_len,
    nbr_fea_len,
    atom_fea_len=20,
    n_conv=1,
    h_fea_len=10,
    n_h=2,
    classification=True,
).to(device)

model


In [ ]:
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
def move_batch(input, target):
    atom_fea, nbr_fea, nbr_fea_idx, crystal_atom_idx = input
    return (
        atom_fea.to(device),
        nbr_fea.to(device),
        nbr_fea_idx.to(device),
        [idx.to(device) for idx in crystal_atom_idx],
    ), target.view(-1).long().to(device)


def train(data_loader):
    model.train()
    total_loss = 0.0

    for input, target, _ in data_loader:
        input_var, target_var = move_batch(input, target)

        optimizer.zero_grad()
        output = model(*input_var)
        loss = criterion(output, target_var)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


In [ ]:
def accuracy(data_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for input, target, _ in data_loader:
            input_var, target_var = move_batch(input, target)
            output = model(*input_var)
            pred = output.argmax(dim=1)
            correct += (pred == target_var).sum().item()
            total += target_var.size(0)

    return correct / total if total else 0.0


In [ ]:
# Kept for reference: for regression notebooks use MAE, but this local sample is classification.
def mae(prediction, target):
    return torch.mean(torch.abs(target - prediction))


In [ ]:
epochs = 5
t_l = []
t_a = []
v_a = []

np.random.seed(1)
torch.manual_seed(1)

for epoch in range(epochs):
    train_loss = train(train_loader)
    train_acc = accuracy(train_loader)
    val_acc = accuracy(val_loader)

    t_l.append(train_loss)
    t_a.append(train_acc)
    v_a.append(val_acc)

    print(f"Epoch: {epoch:02d}, loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, val_acc: {val_acc:.4f}")


In [ ]:
import matplotlib.pyplot as plt
plt.plot(t_a)
plt.plot(v_a)
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend(["Train", "Valid"])
plt.title("Train vs Valid Accuracy")
plt.show()


In [ ]:
plt.plot(t_l)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend(["Train"])
plt.title("Training Loss")
plt.show()


In [ ]:
y_hat = []
y = []
model.eval()
with torch.no_grad():
    for input, target, _ in test_loader:
        input_var, target_var = move_batch(input, target)
        pred = model(*input_var).argmax(dim=1)
        y_hat.append(pred.cpu())
        y.append(target_var.cpu())


In [ ]:
y_1_hat = y_hat[0].numpy()
y_1 = y[0].numpy()


In [ ]:
print("Predicted:", y_1_hat[:10])
print("Actual:   ", y_1[:10])
print("Test accuracy:", accuracy(test_loader))
